# Ejercicio 1: Diagnostico de un estudiante: ¿aprobará o desaprobará?

Usamos una **red bayesiana** para modelar cómo influyen las horas de estudio, la asistencia y la nota del examen en la aprobación del estudiante.

## 1. Estructura de la red bayesiana

La red es una **cadena** (cada variable depende únicamente de la anterior):

```
HorasEstudio (Pocas / Muchas)
        ↓
Asistencia  (Baja / Alta)
        ↓
NotaExamen  (Baja / Alta)
        ↓
Aprobacion  (Desaprueba / Aprueba)
```

## 2. Tablas de probabilidad condicional (CPD)

**a) Horas de estudio (variable raiz)** — `P(Pocas) = 0.60` → `P(Muchas) = 0.40`

**b) Asistencia | HorasEstudio**

| Asistencia | Pocas | Muchas |
|---|---|---|
| Baja  | 0.80 | 0.30 |
| Alta   | 0.20 | 0.70 |

**c) NotaExamen | Asistencia**

| NotaExamen | Baja | Alta |
|---|---|---|
| Baja | 0.70 | 0.20 |
| Alta | 0.30 | 0.80 |

**d) Aprobacion | NotaExamen**

| Aprobacion | Baja | Alta |
|---|---|---|
| Desaprueba | 0.95 | 0.10 |
| Aprueba    | 0.05 | 0.90 |

## 3. ¿Cómo se calcula?

Para cualquier consulta se aplica la regla de Bayes y la descomposicion segun la estructura:

$$P(A \mid E) = \frac{P(A, E)}{P(E)}$$

donde `A` = variable de interes y `E` = evidencias. La red usa la factorizacion:

$$P(H, A, N, P) = P(H) \cdot P(A \mid H) \cdot P(N \mid A) \cdot P(P \mid N)$$

con `H = HorasEstudio`, `A = Asistencia`, `N = NotaExamen`, `P = Aprobacion`.

In [ ]:
#!pip install pgmpy

In [1]:
# libreria de redes bayesianas
from pgmpy.models import DiscreteBayesianNetwork
# crear tablas de probabilidades condicionales
from pgmpy.factors.discrete import TabularCPD
# realizar inferencia en la red bayesiana
from pgmpy.inference import VariableElimination

# utilidades para el calculo manual de Bayes
from itertools import product

c:\Users\yerym\OneDrive\Documentos\Unamba\26-II\Fundamentos de IA\Scritps\LangGraph\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\yerym\OneDrive\Documentos\Unamba\26-II\Fundamentos de IA\Scritps\LangGraph\.venv\Lib\site-packages\pgmpy\estimators\__init__.py:4: FutureWarning: `pgmpy.estimators.StructureScore` is deprecated and will be removed in v1.3.0. Use `pgmpy.structure_score` instead.
  from .StructureScore import (


In [2]:
# crear la red bayesiana (cadena)
model = DiscreteBayesianNetwork([
    ("HorasEstudio", "Asistencia"),
    ("Asistencia", "NotaExamen"),
    ("NotaExamen", "Aprobacion"),
])

In [3]:
# CPD de HorasEstudio: variable raiz, P(Pocas)=0.60 y P(Muchas)=0.40
cpd_horas = TabularCPD(
    variable="HorasEstudio",
    variable_card=2,
    values=[[0.6], [0.4]],
    state_names={"HorasEstudio": ["Pocas", "Muchas"]},
)

In [4]:
# CPD de Asistencia dado HorasEstudio
# filas: Baja, Alta   |   columnas: Pocas, Muchas
cpd_asistencia = TabularCPD(
    variable="Asistencia",
    variable_card=2,
    values=[[0.8, 0.3], [0.2, 0.7]],
    evidence=["HorasEstudio"],
    evidence_card=[2],
    state_names={
        "Asistencia": ["Baja", "Alta"],
        "HorasEstudio": ["Pocas", "Muchas"],
    },
)

In [5]:
# CPD de NotaExamen dado Asistencia
# filas: Baja, Alta   |   columnas: Baja, Alta
cpd_nota = TabularCPD(
    variable="NotaExamen",
    variable_card=2,
    values=[[0.7, 0.2], [0.3, 0.8]],
    evidence=["Asistencia"],
    evidence_card=[2],
    state_names={
        "NotaExamen": ["Baja", "Alta"],
        "Asistencia": ["Baja", "Alta"],
    },
)

In [6]:
# CPD de Aprobacion dado NotaExamen
# filas: Desaprueba, Aprueba   |   columnas: Baja, Alta
cpd_aprobacion = TabularCPD(
    variable="Aprobacion",
    variable_card=2,
    values=[[0.95, 0.10], [0.05, 0.90]],
    evidence=["NotaExamen"],
    evidence_card=[2],
    state_names={
        "Aprobacion": ["Desaprueba", "Aprueba"],
        "NotaExamen": ["Baja", "Alta"],
    },
)

In [7]:
# agregar las CPDs al modelo y verificar que sea valido
model.add_cpds(cpd_horas, cpd_asistencia, cpd_nota, cpd_aprobacion)
model.check_model()

True

In [8]:
print("Modelo de red bayesiana creado y verificado correctamente.")
print("Aristas:", model.edges())

Modelo de red bayesiana creado y verificado correctamente.
Aristas: [('HorasEstudio', 'Asistencia'), ('Asistencia', 'NotaExamen'), ('NotaExamen', 'Aprobacion')]


In [9]:
# motor de inferencia sobre el grafo
inferencia = VariableElimination(model)

In [10]:
# funcion auxiliar para consultar la red de forma legible
def consultar_red(variable, evidencia=None, titulo="Consulta"):
    print("\n" + "*" * 40)
    print(titulo)
    print("*" * 40)
    resultado = inferencia.query(
        variables=[variable],
        evidence=evidencia,
        show_progress=False,
    )
    print(resultado)
    return resultado

## 4. Ejemplo de calculo paso a paso

> **Nota importante sobre el enunciado.**
> La infografia propone `P(Aprueba | Muchas, Alta) = 0.30 x 0.80 + 0.70 x 0.10 = 0.31`.
> Ese calculo es **incorrecto**: mezcla las columnas de la tabla `NotaExamen | Alta`
> (que son `0.20 / 0.80`) con los valores de `Asistencia = Baja` (`0.70 / 0.30`),
> y ademas usa `P(Desaprueba | Nota)` en lugar de `P(Aprueba | Nota)`.

> El calculo correcto, con la tabla `NotaExamen | Asistencia Alta` y multiplicando por
> `P(Aprueba | NotaExamen)`, da **0.73**.

In [11]:
# calculo manual de Bayes, sin usar la inferencia automatica
P_H = {"Pocas": 0.60, "Muchas": 0.40}
P_A = {
    "Pocas": {"Baja": 0.80, "Alta": 0.20},
    "Muchas": {"Baja": 0.30, "Alta": 0.70},
}
P_N = {
    "Baja": {"Baja": 0.70, "Alta": 0.30},
    "Alta": {"Baja": 0.20, "Alta": 0.80},
}
P_P = {
    "Baja": {"Desaprueba": 0.95, "Aprueba": 0.05},
    "Alta": {"Desaprueba": 0.10, "Aprueba": 0.90},
}

evidencia = {"HorasEstudio": "Muchas", "Asistencia": "Alta"}
h, a = evidencia["HorasEstudio"], evidencia["Asistencia"]

# pesos P(H, A, N): parte de la evidencia, marginalized sobre NotaExamen
pesos = {n: P_H[h] * P_A[h][a] * P_N[a][n] for n in P_N[a]}

# P(P | H, A) = suma_N  P(P | N) * P(N | A, H)
conjunto = {
    p: sum(pesos[n] * P_P[n][p] for n in pesos)
    for p in ["Desaprueba", "Aprueba"]
}
normalizador = sum(conjunto.values())

print(f"Pesos P(H={h}, A={a}, N):", {k: round(v, 4) for k, v in pesos.items()})
print("Distribucion de Aprobacion:")
for p, v in conjunto.items():
    print(f"  {p:<11} = {v / normalizador:.4f}")

Pesos P(H=Muchas, A=Alta, N): {'Baja': 0.056, 'Alta': 0.224}
Distribucion de Aprobacion:
  Desaprueba  = 0.2700
  Aprueba     = 0.7300


In [12]:
# P(Aprueba | N=Baja, A=Alta) * P(N=Baja | A=Alta) + P(Aprueba | N=Alta, A=Alta) * P(N=Alta | A=Alta)
print("Desglose del resultado correcto:")
for n in ["Baja", "Alta"]:
    print(f"  P(Aprueba | N={n}) * P(N={n} | A=Alta) = {P_P[n]['Aprueba']} * {P_N['Alta'][n]}")
print(f"  Total = {P_P['Baja']['Aprueba'] * P_N['Alta']['Baja'] + P_P['Alta']['Aprueba'] * P_N['Alta']['Alta']:.2f}")

Desglose del resultado correcto:
  P(Aprueba | N=Baja) * P(N=Baja | A=Alta) = 0.05 * 0.2
  P(Aprueba | N=Alta) * P(N=Alta | A=Alta) = 0.9 * 0.8
  Total = 0.73


In [13]:
# pregunta principal del ejercicio
consultar_red(
    "Aprobacion",
    {"HorasEstudio": "Muchas", "Asistencia": "Alta"},
    "Muchas horas de estudio + asistencia alta",
)


****************************************
Muchas horas de estudio + asistencia alta
****************************************
+------------------------+-------------------+
| Aprobacion             |   phi(Aprobacion) |
+========================+===================+
| Aprobacion(Desaprueba) |            0.2700 |
+------------------------+-------------------+
| Aprobacion(Aprueba)    |            0.7300 |
+------------------------+-------------------+


<DiscreteFactor representing phi(Aprobacion:2) at 0x1fd349f3d40>

In [14]:
# probabilidad previa, sin ninguna evidencia
consultar_red("Aprobacion", titulo="Aprobacion sin evidencia")


****************************************
Aprobacion sin evidencia
****************************************
+------------------------+-------------------+
| Aprobacion             |   phi(Aprobacion) |
+========================+===================+
| Aprobacion(Desaprueba) |            0.5250 |
+------------------------+-------------------+
| Aprobacion(Aprueba)    |            0.4750 |
+------------------------+-------------------+


<DiscreteFactor representing phi(Aprobacion:2) at 0x1fd34ea89b0>

In [15]:
# efecto de la asistencia sobre la nota del examen
consultar_red("NotaExamen", {"Asistencia": "Alta"}, "NotaExamen dado asistencia alta")


****************************************
NotaExamen dado asistencia alta
****************************************
+------------------+-------------------+
| NotaExamen       |   phi(NotaExamen) |
+==================+===================+
| NotaExamen(Baja) |            0.2000 |
+------------------+-------------------+
| NotaExamen(Alta) |            0.8000 |
+------------------+-------------------+


<DiscreteFactor representing phi(NotaExamen:2) at 0x1fd3628d580>

In [16]:
# inferencia hacia atras (diagnostico): que asistencia tuvo un estudiante que aprobo
consultar_red("Asistencia", {"Aprobacion": "Aprueba"}, "Asistencia dado que el estudiante aprueba")


****************************************
Asistencia dado que el estudiante aprueba
****************************************
+------------------+-------------------+
| Asistencia       |   phi(Asistencia) |
+==================+===================+
| Asistencia(Baja) |            0.3853 |
+------------------+-------------------+
| Asistencia(Alta) |            0.6147 |
+------------------+-------------------+


<DiscreteFactor representing phi(Asistencia:2) at 0x1fd3628daf0>

In [17]:
# diagnostico completo: horas de estudio de quien aprueba con nota alta
consultar_red(
    "HorasEstudio",
    {"Aprobacion": "Aprueba", "NotaExamen": "Alta"},
    "HorasEstudio dado que aprueba con nota alta",
)


****************************************
HorasEstudio dado que aprueba con nota alta
****************************************
+----------------------+---------------------+
| HorasEstudio         |   phi(HorasEstudio) |
+======================+=====================+
| HorasEstudio(Pocas)  |              0.4800 |
+----------------------+---------------------+
| HorasEstudio(Muchas) |              0.5200 |
+----------------------+---------------------+


<DiscreteFactor representing phi(HorasEstudio:2) at 0x1fd34ec5d60>

## 5. Conclusion

La red bayesiana combina la informacion de todas las variables y devuelve la probabilidad
final de aprobar o desaprobar, actualizando las probabilidades a medida que se agrega evidencia.

- Con **muchas horas + asistencia alta** la probabilidad de **aprobar** sube a **0.73**.
- Sin evidencia, la probabilidad de aprobar queda cerca de **0.475** (ligeramente en contra del estudiante).
- El motor `VariableElimination` recorre el grafo en ambas direcciones, por eso tambien permite
  el diagnostico inverso (de *Aprobacion* hacia *Asistencia* u *HorasEstudio*).